## Summary Table: All Ollama Usage Modes

| Mode | Best For | Example |
|------|----------|---------|
| **CLI** | Testing, experimentation | `ollama run llama3.2:1b` |
| **Python Library** | Building applications | `ollama.generate(model='...', prompt='...')` |
| **REST API** | Custom integrations | `POST http://localhost:11434/api/generate` |
| **LangChain** | Complex applications with multiple components | `ChatOllama(model='...')` |
| **Ollama Cloud** | Running large parameter models | `ollama run deepseek-v3:671b` |
| **Ollama App** | Simple chat interface | GUI chat application |

---

## Quick Reference Commands

```bash
# Installation & Setup
ollama --version                    # Check installation
ollama pull model_name              # Download a model
ollama list                         # List downloaded models

# Running Models
ollama run model_name               # Start interactive session
ollama run model_name "prompt"      # One-shot prompt

# Management
ollama rm model_name                # Delete a model
ollama cp source destination        # Copy a model
ollama show model_name              # Show model details

# Cloud
ollama sign in                      # Sign in to Ollama Cloud
ollama sign out                     # Sign out
```

---

## Key Takeaways

1. **Ollama simplifies open source LLM usage** - handles all technical complexities
2. **Multiple usage modes** - choose based on your needs (CLI for testing, Python for apps)
3. **Tool calling** - extends LLM capabilities to interact with external systems
4. **Model files** - customize behavior without retraining
5. **LangChain integration** - build production applications with orchestration
6. **Ollama Cloud** - access large models without powerful hardware
7. **Privacy trade-off** - local models for privacy, cloud models for larger capabilities


## Topic 8: Tool Calling (Function Calling)

### Explanation
Tool calling allows LLMs to use external tools/systems to perform tasks they cannot do by themselves. This is how LLMs can interact with databases, APIs, calculate things, etc.

### Why Tool Calling?
LLMs have limitations:
- Knowledge cutoff date
- Cannot interact with external systems
- Cannot perform calculations
- Cannot access databases

### How Tool Calling Works

```
1. CREATE TOOLS (Python functions)
   ↓
2. CREATE TOOL SCHEMA (JSON describing tools)
   ↓
3. CALL LLM WITH PROMPT + TOOL SCHEMA
   ↓
4. LLM RETURNS JSON (which tool to call + parameters)
   ↓
5. MANUALLY EXECUTE THE TOOL
   ↓
6. PASS RESULT BACK TO LLM
   ↓
7. LLM GENERATES FINAL RESPONSE
```
<img src="Screenshot 2026-09-07 at 12.25.45 AM.png">


### Important: Only models with tool calling capability can do this.

In [1]:
import ollama 
import json

## ---a dummy dataset --- ##
inventory_db={
    "laptop":{"stock":5,"baseprice":1200},
    "monitor":{"stock":0,"baseprice":300},
    "iphone": {"stock": 10, "baseprice": 800}
}


# --------- Step 1 : Define Tool functions (NORMAL PYTHON FUNCTIONS)--------
# Tool function 1 (checking database)
def check_inventory(product_name):
    """CHECK IF PRODUCT IS IN INVENTORY"""
    product_name=product_name.lower()

    # if product found
    if product_name in inventory_db:
        return inventory_db[product_name]
    
    # if product not found
    return {"stock":0,"baseprice":None}

# Tool function 2 (Business logic for discounts)
def calculate_loyalty_discount(base_price,years_as_customers):
    """CALCULATE DISCOUNT BASED ON CUSTOMERS LOYALTY"""
    discount=min(years_as_customers*0.05,0.30) # max 30% discount
    final_price=base_price*(1-discount)
    return round(final_price,2)



# ---------------STEP 2:CREATE TOOL SCHEMAS (JSON DESCRIBING THE TOOLS)----------------
tools=[
    # for tool 1
    {
        "type":"function",
        "function":{
            "name":"check_inventory",
            "description":"Get stock and price for a product",
            "parameters":{
                "type":"object",
                "properties":{
                    "product_name":{"type":"string"},
                },
            "required":["product_name"]
            },
            
        }
    },

    # for tool 2
    {
        "type":"function",
        "function":{
            "name":"calculate_loyalty_discount",
            "description":"Calculate final price based on loyalty years",
            "parameters":{
                "type":"object",
                "properties":{
                    "base_price":{"type":"number"},
                    "years_as_customers":{"type":"integer"}
                },
            "required":["base_price","years_as_customers"]
            },
            
        }
    }
]






In [2]:
tools

[{'type': 'function',
  'function': {'name': 'check_inventory',
   'description': 'Get stock and price for a product',
   'parameters': {'type': 'object',
    'properties': {'product_name': {'type': 'string'}},
    'required': ['product_name']}}},
 {'type': 'function',
  'function': {'name': 'calculate_loyalty_discount',
   'description': 'Calculate final price based on loyalty years',
   'parameters': {'type': 'object',
    'properties': {'base_price': {'type': 'number'},
     'years_as_customers': {'type': 'integer'}},
    'required': ['base_price', 'years_as_customers']}}}]

In [3]:
# Map function names to actual functions
function_map={
    "check_inventory":check_inventory,
    "calculate_loyalty_discount":calculate_loyalty_discount
}


In [4]:

# ---------------------STEP 3: START A CONVERSATION-----------------
#"" I want to buy monitor can u check stock?"
# "I am a customer for 5 years.What will be the final price of a laptop?"
# also this is used for storing conversations
messages=[
    {
        "role":"user",
        "content":"I am a customer for 5 years. What will be the final price of a laptop?"
    }
]


# ----------- STEP 4:FIRST LLM CALL--------------------
response=ollama.chat(
    model="qwen3:8b", #Must have tool calling,thinking ability
    messages=messages,
    tools=tools
)

print(response)

model='qwen3:8b' created_at='2026-09-08T05:30:21.461979Z' done=True done_reason='stop' total_duration=8983284208 load_duration=3990041 prompt_eval_count=209 prompt_eval_duration=78426000 eval_count=332 eval_duration=8853386000 message=Message(role='assistant', content='', thinking='Okay, let\'s tackle this user query. The user says, "I am a customer for 5 years. What will be the final price of a laptop?" They want to know the final price after applying their loyalty discount.\n\nFirst, I need to check what functions are available. There\'s \'check_inventory\' to get stock and price for a product, and \'calculate_loyalty_discount\' to calculate the final price based on loyalty years. \n\nThe user mentioned being a customer for 5 years, so the loyalty years are 5. But they didn\'t specify the base price of the laptop. Wait, the \'calculate_loyalty_discount\' function requires both base_price and years_as_customers. Since the user hasn\'t provided the base price, maybe I need to first che

In [5]:
response["message"] # or response.message

Message(role='assistant', content='', thinking='Okay, let\'s tackle this user query. The user says, "I am a customer for 5 years. What will be the final price of a laptop?" They want to know the final price after applying their loyalty discount.\n\nFirst, I need to check what functions are available. There\'s \'check_inventory\' to get stock and price for a product, and \'calculate_loyalty_discount\' to calculate the final price based on loyalty years. \n\nThe user mentioned being a customer for 5 years, so the loyalty years are 5. But they didn\'t specify the base price of the laptop. Wait, the \'calculate_loyalty_discount\' function requires both base_price and years_as_customers. Since the user hasn\'t provided the base price, maybe I need to first check the inventory to get the laptop\'s price. \n\nSo, I should call \'check_inventory\' with product_name as "laptop" to retrieve the base price. Once I have that, then use \'calculate_loyalty_discount\' with the base price and 5 years.

In [6]:
response.message.tool_calls

[ToolCall(function=Function(name='check_inventory', arguments={'product_name': 'laptop'}))]

In [7]:
# -----------  Step 5: Check if tool needs to be called -------------


# Check if the model asked to use any tools
tool_calls = response.message.tool_calls


if tool_calls:
    # for each tool
    for tool_call in tool_calls:
        # get the function name and arguments
        tool_name = tool_call['function']['name']
        tool_args = tool_call['function']['arguments']

         # Execute the function
        if tool_name in function_map:
            # Get the actual Python function
            function_to_call=function_map[tool_name]
            # Run the function with given arguments
            result=function_to_call(**tool_args)

            # Add the model's tool request to conversation
            messages.append(response.message)

            #Also add tool result back to conversation
            messages.append(
                {
                    "role":"tool",
                    "content":str(result)
                }
            )

messages


[{'role': 'user',
  'content': 'I am a customer for 5 years. What will be the final price of a laptop?'},
 Message(role='assistant', content='', thinking='Okay, let\'s tackle this user query. The user says, "I am a customer for 5 years. What will be the final price of a laptop?" They want to know the final price after applying their loyalty discount.\n\nFirst, I need to check what functions are available. There\'s \'check_inventory\' to get stock and price for a product, and \'calculate_loyalty_discount\' to calculate the final price based on loyalty years. \n\nThe user mentioned being a customer for 5 years, so the loyalty years are 5. But they didn\'t specify the base price of the laptop. Wait, the \'calculate_loyalty_discount\' function requires both base_price and years_as_customers. Since the user hasn\'t provided the base price, maybe I need to first check the inventory to get the laptop\'s price. \n\nSo, I should call \'check_inventory\' with product_name as "laptop" to retrieve

In [8]:
# ---------------- STEP 6:SECOND CALL LLM WITH CHAT HISTORY -------------------
second_response=ollama.chat(
    model="qwen3:8b", 
    messages=messages,
    tools=tools
)


print(second_response)

model='qwen3:8b' created_at='2026-09-08T05:30:28.793187Z' done=True done_reason='stop' total_duration=7305249541 load_duration=1151125 prompt_eval_count=567 prompt_eval_duration=1042983000 eval_count=236 eval_duration=6254329000 message=Message(role='assistant', content='', thinking='Okay, the user wants to know the final price of a laptop after applying their loyalty discount. They mentioned being a customer for 5 years. \n\nFirst, I called the check_inventory function for "laptop" and got the base price of $1200. Now, I need to calculate the loyalty discount. The function calculate_loyalty_discount requires base_price and years_as_customers. \n\nThe base_price is 1200, and years_as_customers is 5. I should call that function with these values. Let me make sure the parameters are correct. The function will handle the discount calculation based on the years. \n\nOnce I get the discounted price from the function, I can present it to the user. I need to structure the tool call properly w

## from the output we can see the model still needs the tools ,tool call to get to the final answer

In [9]:
# Execute the second tool call
tool_calls = second_response.message.tool_calls

if tool_calls:
    for tool_call in tool_calls:
        tool_name = tool_call['function']['name']
        tool_args = tool_call['function']['arguments']
        
        if tool_name in function_map:
            function_to_call = function_map[tool_name]
            result = function_to_call(**tool_args)
            
            messages.append(second_response.message)
            messages.append({
                "role": "tool",
                "content": str(result)
            })

# Third LLM call to get final answer
final_answer = ollama.chat(
    model="qwen3:8b",
    messages=messages
)

print(final_answer.message.content)

The final price of the laptop after applying your 5-year customer loyalty discount is **$900.00**. 

This is calculated from the original base price of **$1,200** with a discount applied for your long-term loyalty. Let me know if you'd like further details! 😊


## Recommended way ,to use loop till the model wants the tools to get to the final answer

In [10]:
messages = [
    {
        "role": "user",
        "content": "I am a customer for 5 years. What will be the final price of a laptop?"
    }
]

while True:
    # Call LLM
    response = ollama.chat(
        model="qwen3:8b",
        messages=messages,
        tools=tools
    )
    
    # Add assistant response to messages
    messages.append(response.message)
    
    # Check if model wants to call tools
    tool_calls = response.message.tool_calls
    
    if tool_calls:
        # Execute each tool call
        for tool_call in tool_calls:
            tool_name = tool_call['function']['name']
            tool_args = tool_call['function']['arguments']
            
            if tool_name in function_map:
                result = function_map[tool_name](**tool_args)
                
                # Add tool result to messages
                messages.append({
                    "role": "tool",
                    "content": str(result)
                })
    else:
        # No more tool calls - model has final answer
        print(response.message.content)
        break

The final price of the laptop after applying your 5-year loyalty discount is **$900.00**. There are 5 units available in stock! 🛍️
